In [16]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`


In [17]:
using LinearAlgebra
using SparseArrays
using Convex
using GLPK, MathOptInterface, JuMP

In [18]:
### Constants
const μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
const J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)const 
const Re_mks = 6378.137 #Radius of the Earth (km)
const Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

const Cd = 2.2 #Drag coefficient (unitless)
const AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
const AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [19]:
#Orbital Parameters (roughly ISS)
const a_mks = 6791.0 #semi-major axis (km)
const inc = 54.74 #inclination (deg)
const n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
const P_mks = 2*pi/n_mks; #orbital period (sec) 1.54 hour 5569.440609858254 sec


In [20]:
#Normalize units
const a = 1.0 #distance unit is semi-major axis
const P = 1 #time unit is orbital period
const μ = (2*pi/P)^2 * a^3 #μ_mks*(P_mks^2/a_mks^3)
const n = sqrt(μ/a^3)
const Re = Re_mks/a_mks
const Ωe = Ωe_mks*P_mks;

const AoM = AoM_mks/(a_mks^2)
const AoM0 = AoM0_mks/(a_mks^2);


In [21]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    θ = atan(x[2],x[1]) #azimuth
    ϕ = atan(sqrt(x[1]^2+x[2]^2),x[3]) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_θaΩ(x)
    r = x[1:3]
    v = x[4:6]
        
    E = 0.5*v'*v + potential(cartesian_to_spherical(x))
    
    a = -μ/(2*E)
    
    i_x = [1,0,0]
    i_y = [0,1,0]
    i_z = [0,0,1]
    
    h = cross(r,v)
    Ω = atan(h[2]/norm(h[1:2]), h[1]/norm(h[1:2]))
    
    n = cross(i_z,h)/norm(cross(i_z,h))
    if dot(r,i_z) > 0
        θ = acos(dot(n,(r/norm(r))))
    else
        θ = 2*pi- acos(dot(n,(r/norm(r))))
    end
    return [θ; a; Ω]
end

function cartesian_to_ΔθaΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(cartesian_to_spherical(x0))
    E1 = 0.5*v1'*v1 + potential(cartesian_to_spherical(x1))
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    

    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))

    
    h0 = cross(r0,v0)
    Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
    Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    
    ΔΩ = Ω1-Ω0
    
    return [Δθ; Δa; ΔΩ]
end

function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [22]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [23]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1
function kvalues(a)
    n = sqrt(μ/a^3)
    Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2*0.965#-15/a_mks
    ȧ = 2*(a^2/μ)*(a)*n*Fd
    k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
    k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
    k3 = ȧ #dȧ/d(Drag) - 3*sqrt(μ*a)*J2*Re*Re/a^3*sind(inc)^2 
    k4 = k1/k2
    return [k1, k2, k3, k4]
end

kvalues (generic function with 1 method)

In [24]:
#Discrete-time model
num_sat = 3 # number of couple of satellites 


function create_continuous(a, num_sat)
    #Discrete-time model
#     k1, k2, k3, k4 = kvalues(a)
    A_bar = zeros(3*num_sat,3*num_sat) 
    B_bar = zeros(3*num_sat,num_sat+1) 

    # B_bar[1:3,1:2]=B0
    for k =0:num_sat-1
        k1, k2, k3, k4 = kvalues(a[k+1])
        A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = [0 k2 0; 0 0 0; 0 k1 0]
        B_bar[3*k+2,1] = -k3
        B_bar[3*k+2,2+k] = k3
    end
    return A_bar, B_bar 
end

function c2d(a,dt,num_sat)
    # discretize a continuous time linear system 
    # convert xdot = Ax + Bu 
    # into x_{k+1} = Ad x_k + Bd x_k 
    # where we have dt between k and k +1 
    
    A,B = create_continuous(a, num_sat)
    nx,nu = size(B)
    
    H = exp(dt*[A B;zeros(nu, nx + nu)])
    Ad = H[1:nx, 1:nx]
    Bd = H[1:nx, (nx + 1):(nx + nu)]
    return Ad, Bd 
end

a_vec = a*ones(num_sat)
const h =1 #time step
Ad, Bd = c2d(a_vec, h, num_sat);


In [25]:
function sat_dynamics_rk4(x,u,h)
    f1 = satellite_dynamics(x,u)
    f2 = satellite_dynamics(x + 0.5*h*f1,u)
    f3 = satellite_dynamics(x + 0.5*h*f2, u)
    f4 = satellite_dynamics(x + h*f3, u)
    return x + (h/6.0)*(f1 + 2*f2 +2*f3 +f4)
end

sat_dynamics_rk4 (generic function with 1 method)

In [26]:
### Full Orbital Dynamics with J2 and Drag
function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end


function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r1 = x[1:3]
    v1 = x[4:6]
    r2 = x[7:9]
    v2 = x[10:12]
    u1 = u[1]
    u2 = u[2]

    
    pos1_x = r1[1]
    pos1_y = r1[2]
    pos1_z = r1[3]
    r1_norm = sqrt(r1[1]^2+r1[2]^2+r1[3]^2)
    
    pos2_x = r2[1]
    pos2_y = r2[2]
    pos2_z = r2[3]
    r2_norm = sqrt(r2[1]^2+r2[2]^2+r2[3]^2)
    
    g1 = +1.5*J2*μ_mks*Re_mks^2/r1_norm^4*[
        pos1_x/r1_norm*(5*pos1_z^2/r1_norm^2-1);
        pos1_y/r1_norm*(5*pos1_z^2/r1_norm^2-1);
        pos1_z/r1_norm*(5*pos1_z^2/r1_norm^2-3)] - μ_mks*r1/(r1_norm^3)
    
    g2 = +1.5*J2*μ_mks*Re_mks^2/r2_norm^4*[
        pos2_x/r2_norm*(5*pos2_z^2/r2_norm^2-1);
        pos2_y/r2_norm*(5*pos2_z^2/r2_norm^2-1);
        pos2_z/r2_norm*(5*pos2_z^2/r2_norm^2-3)] - μ_mks*r2/(r2_norm^3)
    
    
    v1_rel = v1 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ1 = density_mks(r1_norm - Re_mks)
    Fdrag1 = -0.5*(u1*AoM_mks)*Cd*ρ1*(norm(v1_rel))*v1_rel
    
    v2_rel = v2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ2 = density_mks(r2_norm - Re_mks)
    Fdrag2 = -0.5*(u2*AoM_mks)*Cd*ρ2*(norm(v2_rel))*v2_rel

    v̇1 = g1 + Fdrag1#- μ_mks*r1/(r1_norm^3) + Fdrag1 #g1
    v̇2 = g2 + Fdrag2#- μ_mks*r2/(r2_norm^3) + Fdrag2 #g2 
    return [v1; v̇1; v2; v̇2]
end


satellite_dynamics (generic function with 1 method)

In [27]:
#Initial and terminal constraints and #Inequality constraints
params = (
    l_12 = 1.0, # number of revolutions
    l_13 = 2.0,
    l_14 = 3.0,
    l_15 = 4.0,
    l_16 = 5.0,
    Δx0 = zeros(size(Ad,1)),
    Δaf = 0, #end at the same altitude
    u_min = 0,
    u_max = 1,
    Δa_min = -10/a_mks, # /num_sat
    Δa_max = 10/a_mks,) #/num_sat 

k4 = kvalues(a)[4]
params = (; params..., Δθf = 2*pi*[params.l_12,params.l_13,params.l_14,params.l_15,params.l_16],
)

params = (; params..., ΔΩf = k4*(params.Δθf))# Right Ascension Separation


(l_12 = 1.0, l_13 = 2.0, l_14 = 3.0, l_15 = 4.0, l_16 = 5.0, Δx0 = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Δaf = 0, u_min = 0, u_max = 1, Δa_min = -0.0014725371815638344, Δa_max = 0.0014725371815638344, Δθf = [6.283185307179586, 12.566370614359172, 18.84955592153876, 25.132741228718345, 31.41592653589793], ΔΩf = [-0.012123811564198254, -0.024247623128396508, -0.03637143469259476, -0.048495246256793016, -0.06061905782099127])

In [28]:
#Discretized Optimal Control Problem Parameters
const Nx = size(Ad,1)     # number of state
const Nu = size(Bd,2)     # number of controls
const Tfinal = 1600#Int(round(800*P_mks)) # final time (number of orbits)
const Nt = Int(Tfinal/h)+1    # number of time steps
const Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));

#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]
U1 = [sparse(kron(Diagonal(I,Nt), sparse([I zeros(1,(Nu-1+Nx))]))) for i = 1:num_sat] #Matrix that picks out all u of sat 1
U2 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,i) I zeros(1,(Nu-i)+Nx-1)]))) for i = 1:num_sat] #Matrix that picks out all u of sat n
X1 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-1-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x1 = Δθ
X2 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,1) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-2-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x2 = Δa
X3 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,2) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-3-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x3 = ΔΘ


X1F = spzeros(num_sat,(Nu+Nx)*Nt);  #Matrix that picks out x1f = Δθf
X2F = spzeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x2f = Δaf
# X3F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x3f = ΔΘf

for i = num_sat:-1:1
    X1F[num_sat-i+1,end-(i)*3+1] = 1.0
    X2F[num_sat-i+1,end-(i)*3+2] = 1.0
#     X3F[num_sat-i+1,end-(i)*3+3] = 1.0
end


In [29]:
using MosekTools
function LP_controller(a,Δx, params)
    z = Variable(Nz)
    
    #Dynamics equality constraints C*z = d
    Ad, Bd = c2d(a, h, num_sat)
    C = sparse([[Bd -I zeros(Nx,(Nt-1)*(Nu+Nx))]; zeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [Ad Bd]) zeros((Nt-1)*Nx,Nx)] + [zeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[zeros(Nx,Nu) Diagonal(-I,Nx)])]])
    d = [-Ad*Δx; zeros(Nx*(Nt-1))];

    #Convex Formulation
    objective = 0
    Ac = U1[1] 
    b = zeros(Nt + Nt*num_sat + Nt*num_sat)
    for i in 1:num_sat
        Ac  = [Ac; U2[i];]
    end

    for i in 1:num_sat
        Ac  = [Ac; X1[i]]
        b[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
    end
    objective += norm(Ac*z-b,1)# + sum(norm(X1[:,:,i]*z - Δθf[i],1) for i in 1:num_sat);

    problem = minimize(objective);
    problem.constraints += [C*z == d] #dynamics constraints

    for i = 1:num_sat
        problem.constraints += [X2F[i,:]'*z == params.Δaf] #final altitude constraint
    #     problem.constraints += [X3F[i,:]'*z == ΔΩf] #final altitude constraint
        problem.constraints += [U1[i]*z >= params.u_min; U1[i]*z <= params.u_max] #drag constraints
        problem.constraints += [U2[i]*z >= params.u_min; U2[i]*z <= params.u_max] #drag constraints
        problem.constraints += [X2[i]*z >= params.Δa_min; X2[i]*z <= params.Δa_max]; #altitude constraints
    end
    
    solve!(problem, Mosek.Optimizer(); silent_solver = true)#Mosek.Optimizer)
    # Optimizer
#     optimizer = GLPK.Optimizer()
#     MOI.set(optimizer, MOI.RawParameter("msg_lev"), GLPK.GLP_MSG_ON)
    #     solve!(problem,optimizer; silent_solver = false)
    
    u1 = zeros(size(U1[1])[1],num_sat) # sat 1 control
    u2 = zeros(size(U2[1])[1],num_sat) # sat n control
    for i = 1:num_sat
        u1[:,i] = U1[i]*z.value;
        u2[:,i] = U2[i]*z.value;
    end
    return [u1[1:h,:],u2[1:h,:]]
end

LP_controller (generic function with 1 method)

In [31]:
using Statistics
# initial conditions
r_s0 = [a_mks; 0; 0]
v_s0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
r_s1 = [a_mks; 0; 0]
v_s1 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = repeat([r_s0; v_s0; r_s1; v_s1],num_sat);

# initialization results
# u1hist = zeros(size(U1[1])[1]*h,num_sat)
# u2hist = zeros(size(U2[1])[1]*h,num_sat)
# Δθhist = zeros(size(X1[1])[1]*h,num_sat)
# Δθunhist = zeros(size(X1[1])[1]*h,num_sat)
# Δahist = zeros(size(X2[1])[1]*h,num_sat)
# ΔΩhist = zeros(size(X3[1])[1]*h,num_sat)
# tp_hist = zeros(1,length(thist)*h)

u1hist = zeros(10000,num_sat)
u2hist = zeros(10000,num_sat)
Δθhist = zeros(10000,num_sat)
Δθunhist = zeros(10000,num_sat)
Δahist = zeros(10000,num_sat)
ΔΩhist = zeros(10000,num_sat)
tp_hist = zeros(1,10000)

h_traj = 10 # integrate traj 1/5 Hz
T = P_mks*h # integrate traj for n orbital period
# x_hist = zeros(length(x0),Int(round(T/h_traj))*Nt+1)
# t_hist = zeros(1,Int(round(T/h_traj))*Nt+1)
x_hist = zeros(length(x0),Int(round(T/h_traj))*10000+1)
t_hist = zeros(1,Int(round(T/h_traj))*10000+1)

# find first controller
xc = zeros(size(Ad,1))
for k = 0:num_sat-1
    xc[1+k*3:3+k*3,1] = cartesian_to_ΔθaΩ(mks_to_orbital(x0[1+k*12:6+k*12]),mks_to_orbital(x0[7+k*12:12+k*12]))
end
uk = LP_controller(a_vec, xc, params)

# store first step
x_hist[:,1] = x0
for k = 0:num_sat-1
    Δθhist[1,k+1] = xc[1+k*3]
    Δahist[1,k+1] = xc[2+k*3]
    ΔΩhist[1,k+1] = xc[3+k*3]
    u1hist[1,k+1] = uk[1][k+1]
    u2hist[1,k+1] = uk[2][k+1]
end

# initialize index
kk = 0 # traj steps count
mm = zeros(num_sat,1) # Wrapping matrix
ll = 0 # orbit counter
for k = 1:10000-1 # control horizon
    ss = 0 # orbit counter for indexing
    for orbit = 1:h # save results for each orbit
        ll += 1 # increase orbit counter
        for time = 0:h_traj:P_mks # loop for simulations for entire orbit
            kk += 1 # increase traj steps counter
            
            for jj = 0:num_sat-1 # loop for simulation for each couple of satellites
                u = [uk[1][jj*h+orbit],uk[2][jj*h+orbit]] # pick control stored
                x_hist[1+jj*12:12+jj*12,kk+1] .= sat_dynamics_rk4(x_hist[1+jj*12:12+jj*12,kk],u,h_traj) #simulate couple of sats
            end
            t_hist[kk+1] = time # save simulation time
        end
        
        for jj = 0:num_sat-1 # transform from cartesian to ΔθaΩ coordinates for each couple of sats
            xc[1+jj*3:3+jj*3,1] = cartesian_to_ΔθaΩ(mks_to_orbital(x_hist[1+jj*12:6+jj*12,kk+1]),mks_to_orbital(x_hist[7+jj*12:12+jj*12,kk+1]))
        end

        #save results
        for jj = 0:num_sat-1
            Δθhist[ll+1,jj+1] = xc[1+jj*3]
            Δahist[ll+1,jj+1] = xc[2+jj*3]
            ΔΩhist[ll+1,jj+1] = xc[3+jj*3]

            u1hist[ll+1,jj+1] = uk[1][jj*h+orbit]
            u2hist[ll+1,jj+1] = uk[2][jj*h+orbit]

#             println("sat num ", jj, " orbit ", orbit, " jj ", jj, " jj*h+orbit ",jj*h+orbit," u1 ",u1hist[ll+1,jj+1], " u2 ", u2hist[ll+1,jj+1])

            # angle wrapping, adjust angle for linearized dynamics of controller
            if abs(Δθhist[ll+1,jj+1]-Δθhist[ll,jj+1]) > pi-0.1
                mm[jj+1] =mm[jj+1]+ 2
                println("mm change ", abs(Δθhist[ll+1,jj+1]-Δθhist[ll,jj+1]), " - ",mm[jj+1] )
            end

            xc[1+jj*3] = mm[jj+1]*pi + xc[1+jj*3]
            Δθunhist[ll+1,jj+1] = xc[1+jj*3]
        end

        tp_hist[ll+1] = ll # store orbit time
        ss += 1*(num_sat-1) # increase index counter
#         println("End Loop - k ", k, " orbit ", orbit, " u = ",[uk[1][orbit],uk[2][orbit]])
    end
    
    # evaluate mean apoapsis between each couple of sats to update linearized dynamics
    a1 = zeros(num_sat,1); a2 = zeros(num_sat,1); a_upd = zeros(num_sat,1)
    for jj = 0:num_sat-1
        a1[jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[1+12*jj:6+12*jj,kk+1]))[2]
        a2[jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[7+12*jj:12+12*jj,kk+1]))[2]
        a_upd[jj+1] = mean([a1[jj+1],a2[jj+1]]) 
    end
    
    # evaluate controller
    uk = LP_controller(a_upd, xc, params)
#     println("k ",k, " a ", a_upd)#" u ", uk, "
    println("k ",k, " - ",maximum(abs.(Δθunhist[ll+1,:].-params.Δθf[1:num_sat])), " - ", maximum(abs.(Δahist[ll+1,:].-params.Δaf[1])))
    if maximum(abs.(Δθunhist[ll+1,:].-params.Δθf[1:num_sat]))<1e-5 && maximum(abs.(Δahist[ll+1,:].-params.Δaf[1]))<1e-4 # abs(Δθunhist[ll+1,:].-params.Δθf[1:num_sat])
        println("Reached Final Conditions")
        # Resize Results
        Δθhist = Δθhist[1:ll+1,:]
        Δahist = Δahist[1:ll+1,:]
        ΔΩhist = ΔΩhist[1:ll+1,:]
        u1hist = u1hist[1:ll+1,:]
        u2hist = u2hist[1:ll+1,:]
        tp_hist = tp_hist[1:ll+1]
        Δθunhist = Δθunhist[1:ll+1,:]
        x_hist = x_hist[:,1:k]
        t_hist = t_hist[:,1:k]
        break 
    end
end



k 1 - 18.84951913561684 - 7.797318978597367e-6
k 2 - 18.849408719722238 - 1.5602029298844577e-5
k 3 - 18.849224602297223 - 2.3414145709055312e-5
k 4 - 18.848966711624218 - 3.1233683010722935e-5
k 5 - 18.848634975824606 - 3.906065605996378e-5
k 6 - 18.848229322857662 - 4.6895079765518766e-5
k 7 - 18.847749680519446 - 5.4736969098523325e-5
k 8 - 18.847195976441437 - 6.258633906430777e-5
k 9 - 18.846568138089285 - 7.044320474791643e-5
k 10 - 18.84586609276133 - 7.830758127846948e-5
k 11 - 18.84508976758711 - 8.617948385092333e-5
k 12 - 18.844239089525814 - 9.405892770431024e-5
k 13 - 18.843313985364563 - 0.00010194592814516401
k 14 - 18.84231438171672 - 0.00010984050053219896
k 15 - 18.841240205020043 - 0.00011774266027908542
k 16 - 18.84009138153475 - 0.00012565242286122213
k 17 - 18.838867837341517 - 0.000133569803809519
k 18 - 18.837569498339338 - 0.0001414948187167253
k 19 - 18.83619629024342 - 0.00014942748322099852
k 20 - 18.834748138582867 - 0.0001573678130333267
k 21 - 18.83322496

k 165 - 17.78786913312703 - 0.0013990328238857597
k 166 - 17.774548093812957 - 0.0014082892670852498
k 167 - 17.761139018630708 - 0.0014175562987243095
k 168 - 17.747641830367066 - 0.001426833944689565
k 169 - 17.73405645219633 - 0.0014361222309409172
k 170 - 17.720382807648296 - 0.0014454211835212005
k 171 - 17.706620820574425 - 0.0014547308285355331
k 172 - 17.692770415112374 - 0.0014640511921307775
k 173 - 17.678831515648785 - 0.0014733823005257385
k 174 - 17.66485306852563 - 0.0014725796274431824
k 175 - 17.650878501827407 - 0.0014725393137040577
k 176 - 17.636904230267927 - 0.0014725372890890265
k 177 - 17.622930101406478 - 0.0014725371874095838
k 178 - 17.6089561349431 - 0.0014725371823092193
k 179 - 17.594982358906044 - 0.001472537182054534
k 180 - 17.581008801387934 - 0.0014725371820477617
k 181 - 17.567035490095545 - 0.0014725371820520916
k 182 - 17.55306245229323 - 0.0014725371820506483
k 183 - 17.539089714767115 - 0.0014725371820529798
k 184 - 17.525117303791824 - 0.00147253

k 326 - 15.54770936303409 - 0.0014725371819813704
k 327 - 15.533822246668537 - 0.0014725371819863664
k 328 - 15.519935113909675 - 0.0014725371819751532
k 329 - 15.506047936045295 - 0.00147253718197482
k 330 - 15.492160683995738 - 0.0014725371819764854
k 331 - 15.478273328350479 - 0.001472537181964273
k 332 - 15.464385839405802 - 0.0014725371819637179
k 333 - 15.450498187203474 - 0.0014725371819628297
k 334 - 15.436610341571075 - 0.001472537181959055
k 335 - 15.422722272162781 - 0.001472537181959388
k 336 - 15.408833948501355 - 0.0014725371819533928
k 337 - 15.394945340020962 - 0.0014725371819537258
k 338 - 15.381056416110601 - 0.0014725371819532818
k 339 - 15.367167146158266 - 0.0014725371819467314
k 340 - 15.353277499595595 - 0.0014725371819431787
k 341 - 15.339387445942922 - 0.0014725371819361843
k 342 - 15.325496954854726 - 0.0014725371819385158
k 343 - 15.31160599616512 - 0.0014725371819354072
k 344 - 15.29771453993386 - 0.0014725371819268585
k 345 - 15.28382255649176 - 0.001472537

k 488 - 13.288725083553182 - 0.0014739706293505073
k 489 - 13.274733952355021 - 0.0014739720811385304
k 490 - 13.26074273789908 - 0.0014739735347145677
k 491 - 13.246751457063164 - 0.0014739749902902277
k 492 - 13.2327601277113 - 0.0014739764480617978
k 493 - 13.218768768686427 - 0.0014739779082486582
k 494 - 13.204777399797418 - 0.0014739793710641935
k 495 - 13.190786041800074 - 0.0014739808367003615
k 496 - 13.176794716372026 - 0.0014739823053931955
k 497 - 13.162803446081327 - 0.001473983777323884
k 498 - 13.148812254348886 - 0.001473985252717247
k 499 - 13.13482116540429 - 0.0014739867317663524
k 500 - 13.120830204235194 - 0.001473988214669042
k 501 - 13.106839396530265 - 0.0014739897016277093
k 502 - 13.092848768615415 - 0.0014739911928415284
k 503 - 13.078858347383878 - 0.0014739926884727028
k 504 - 13.064868160219685 - 0.00147399418872185
k 505 - 13.050878234914796 - 0.0014739956937465104
k 506 - 13.036888599580184 - 0.0014739972037305371
k 507 - 13.02289928255081 - 0.0014739987

k 651 - 11.00732268028113 - 0.0014725371818431476
k 652 - 10.993300392448754 - 0.0014725371818392619
k 653 - 10.979278261027403 - 0.001472537181835265
k 654 - 10.965256318530397 - 0.0014725371818326005
k 655 - 10.95123460136801 - 0.001472537181830047
k 656 - 10.93721314990302 - 0.0014725371818293809
k 657 - 10.923192008478953 - 0.0014725371818283817
k 658 - 10.909171225420478 - 0.0014725371818270494
k 659 - 10.895150853005443 - 0.0014725371818306021
k 660 - 10.881130947408538 - 0.0014725371818312682
k 661 - 10.86711156861632 - 0.0014725371818320454
k 662 - 10.853092780313801 - 0.0014725371818342659
k 663 - 10.839074649742997 - 0.0014725371818412603
k 664 - 10.825057247533854 - 0.0014725371818427035
k 665 - 10.811040647508248 - 0.0014725371818472555
k 666 - 10.797024926457784 - 0.00147253718184992
k 667 - 10.783010163896538 - 0.0014725371818556932
k 668 - 10.768996441789874 - 0.0014725371818680166
k 669 - 10.75498384426075 - 0.0014725371818689048
k 670 - 10.740972457274625 - 0.001472537

k 814 - 8.730803945206663 - 0.0014725371818617994
k 815 - 8.716749868772471 - 0.0014725371818660182
k 816 - 8.702693961249828 - 0.001472537181869682
k 817 - 8.688636294538554 - 0.0014725371818711253
k 818 - 8.674576946735858 - 0.0014725371818654631
k 819 - 8.660516002070757 - 0.001472537181875011
k 820 - 8.646453550803706 - 0.0014725371818830046
k 821 - 8.632389689091253 - 0.0014725371818938848
k 822 - 8.618324518815601 - 0.0014725371818915534
k 823 - 8.60425814737994 - 0.0014725371819008792
k 824 - 8.590190687469962 - 0.0014725371819109823
k 825 - 8.576122256782863 - 0.0014725371819179767
k 826 - 8.562052977724338 - 0.0014725371819227506
k 827 - 8.547982977074978 - 0.0014725371819367394
k 828 - 8.533912385628 - 0.0014725371819422906
k 829 - 8.519841337799452 - 0.0014725371819513944
k 830 - 8.505769971212864 - 0.0014725371819566124
k 831 - 8.491698426260616 - 0.001472537181974487
k 832 - 8.477626845643858 - 0.001472537181978928
k 833 - 8.463555373893502 - 0.0014725371819879207
k 834 - 

k 979 - 6.419928485992498 - 0.0014742729459473747
k 980 - 6.405884573412937 - 0.0014742752714483887
k 981 - 6.3918388999154185 - 0.001474277632434351
k 982 - 6.377791594274386 - 0.0014742799951494856
k 983 - 6.36374279613462 - 0.0014742823651987536
k 984 - 6.349692655147136 - 0.0014742847426603145
k 985 - 6.33564133018783 - 0.0014742871283192072
k 986 - 6.321589018629069 - 0.0014742833704103342
k 987 - 6.307535859462643 - 0.0014742931062653453
k 988 - 6.293482010288718 - 0.00147429415556799
k 989 - 6.2794276911141065 - 0.0014742967886079983
k 990 - 6.2653730652272035 - 0.0014743033497177649
k 991 - 6.251318337633721 - 0.0014743032679372936
k 992 - 6.237263763976308 - 0.0014742970263342814
k 993 - 6.223209486932911 - 0.0014743079290090444
k 994 - 6.209155662431872 - 0.001474308858286144
k 995 - 6.195102519362356 - 0.0014743115944660934
k 996 - 6.181050268794117 - 0.0014743069581130142
k 997 - 6.166999052693056 - 0.0014743179649145954
k 998 - 6.15294901323824 - 0.0014743189275677748
k 99

k 1141 - 4.143245228511105 - 0.0014725371821052713
k 1142 - 4.129243993739983 - 0.0014725371820861755
k 1143 - 4.115247373318178 - 0.0014725371820728528
k 1144 - 4.101255094233302 - 0.0014725371820561994
k 1145 - 4.087266875248565 - 0.0014725371820309974
k 1146 - 4.073282429878343 - 0.001472537182013678
k 1147 - 4.059301469332418 - 0.0014725371820019095
k 1148 - 4.045323705396118 - 0.0014725371819819255
k 1149 - 4.031348853214327 - 0.0014725371819622746
k 1150 - 4.017376633949212 - 0.0014725371819462874
k 1151 - 4.003406775656316 - 0.0014725377123624428
k 1152 - 3.989439017915192 - 0.0014725377016883145
k 1153 - 3.9754731164285015 - 0.0014725378024199598
k 1154 - 3.961508839418613 - 0.0014725377592543776
k 1155 - 3.947545971675151 - 0.0014725379334590283
k 1156 - 3.9335843147067298 - 0.0014725381266612603
k 1157 - 3.919623684653885 - 0.0014725399256230176
k 1158 - 3.9056639306359404 - 0.0014725370128022552
k 1159 - 3.891704926822946 - 0.0014725372052235564
k 1160 - 3.8777465411901506 -

k 1301 - 1.8979023043702874 - 0.0014724476903200667
k 1302 - 1.88392867951055 - 0.001472447690929135
k 1303 - 1.8699605430597295 - 0.0014724476915342066
k 1304 - 1.855997777250579 - 0.0014724476921376128
k 1305 - 1.8420402258090611 - 0.0014724476927330254
k 1306 - 1.8280876940853759 - 0.0014724476933245523
k 1307 - 1.8141399495759138 - 0.0014724476939134146
k 1308 - 1.80019672283564 - 0.0014724476944895093
k 1309 - 1.7862577087750715 - 0.0014724476950483956
k 1310 - 1.7723225683325268 - 0.0014724476955984
k 1311 - 1.7583909305079715 - 0.0014724476961293087
k 1312 - 1.7444627849577827 - 0.0014723687078253
k 1313 - 1.7305376908059174 - 0.0014723656860391454
k 1314 - 1.7166148617015011 - 0.0014723600566169948
k 1315 - 1.702693839936984 - 0.0014723539774703642
k 1316 - 1.68877413553367 - 0.0014723485225976995
k 1317 - 1.6748552425923862 - 0.0014723432081397148
k 1318 - 1.6609366536385188 - 0.0014723368753799537
k 1319 - 1.6470178493079466 - 0.0014723319078080443
k 1320 - 1.633098311515738 

k 1460 - 0.08123892432877611 - 0.0004581386369799656
k 1461 - 0.07691302693176638 - 0.00044592006964172537
k 1462 - 0.0727018366177461 - 0.00043368361171680636
k 1463 - 0.06860577298252934 - 0.0004214296045488286
k 1464 - 0.06462526872569185 - 0.00040915840736954756
k 1465 - 0.06076076522045071 - 0.00039687060548720954
k 1466 - 0.05701270909239042 - 0.0003845665875150672
k 1467 - 0.0533815485108029 - 0.0003722469742161172
k 1468 - 0.049867728568958825 - 0.00035991239537336295
k 1469 - 0.04647169719847 - 0.0003475615689462197
k 1470 - 0.043193916614111316 - 0.00033519196567444176
k 1471 - 0.04003482317351725 - 0.00032280962062813945
k 1472 - 0.03699481108150593 - 0.00031041526120567653
k 1473 - 0.0340742525291482 - 0.0002980098245519702
k 1474 - 0.03127349347057873 - 0.0002855943047748166
k 1475 - 0.02859285028504388 - 0.0002731697604559935
k 1476 - 0.026032606630650434 - 0.0002607373198852958
k 1477 - 0.023593010518290214 - 0.00024829818859917197
k 1478 - 0.021274271601189554 - 0.00023

In [32]:
using Plots
plotlyjs()

mylabel = String[]
mylabel1 = String[]
mylabel2 = String[]
for i = 1:num_sat
    append!(mylabel,["Sat. 1-"*string(i+1)])
    append!(mylabel1,["Sat. 1"])
    append!(mylabel2,["Sat. "*string(i+1)])
end

plot(tp_hist[1:end],u1hist[1:end,1],label=permutedims(mylabel1), linewidth=2) #,ls=:dot
plot!(tp_hist[1:end],u2hist[1:end,:],label=permutedims(mylabel2)) #,ls=:dot
plot!(xlabel="Orbit Number", ylabel = "Control")

WebIO._IJuliaInit()

┌ Warning: backend `PlotlyJS` is not installed.
└ @ Plots /Users/Josephine/.julia/packages/Plots/nuwp4/src/backends.jl:37


In [39]:
plot(t_hist[1:2000:end],x_hist[1,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[2,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[3,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[7,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[8,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[9,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

In [34]:
plot(tp_hist[1:end],broadcast.(rad2deg,Δθunhist[1:end,:]),label=permutedims(mylabel))
plot!(xlabel="Orbit Number", ylabel = "True Anomaly Difference, deg")

In [35]:
plot(tp_hist[1:end],a_mks*Δahist[1:end,:], label=permutedims(mylabel))
plot!(xlabel="Orbit Number", ylabel = "Semi-major axis Difference, km")


In [36]:
plot(tp_hist[1:end],broadcast.(rad2deg,ΔΩhist[1:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "RAAN Difference, deg")

In [37]:
plot(tp_hist[1:end],a_mks*ΔΩhist[1:end,:],label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [38]:
# Final RAAN
broadcast.(rad2deg,ΔΩhist[:,end]')

println("Targeted Δθf ",  broadcast.(rad2deg, params.Δθf[1:num_sat]), " deg" )
println("Reached Δθf ",  broadcast.(rad2deg, Δθunhist[end,:]), " deg" )


println("Targeted ΔΩ ",  broadcast.(rad2deg, params.ΔΩf[1:num_sat]), " deg" )
println("Reached ΔΩ ",  broadcast.(rad2deg, ΔΩhist[end,:]), " deg" )
println("Final Error ΔΩ ", broadcast.(rad2deg, (params.ΔΩf[1:num_sat] .- ΔΩhist[end,:])), " deg" )
# println("Final Azimut Sat 1 ", broadcast.(rad2deg, ϕ_soln1[:,end]), " deg")
# println("Final Azimut Sat Coupled ", broadcast.(rad2deg, ϕ_soln2[:,end]), " deg")

# println("Final Elevation Sat 1 ", broadcast.(rad2deg, θ_soln1[:,end]), " deg")
# println("Final Elevation Sat Coupled ", broadcast.(rad2deg, θ_soln2[:,end]), " deg")

Targeted Δθf [360.0, 720.0, 1080.0] deg
Reached Δθf [359.9997878814892, 720.0036738641974, 1079.9978839251078] deg
Targeted ΔΩ [-0.6946432342404609, -1.3892864684809219, -2.0839297027213823] deg
Reached ΔΩ [-0.6982570274510985, -1.3984719333329054, -2.100642734056999] deg
Final Error ΔΩ [0.0036137932106376537, 0.009185464851983769, 0.01671303133561636] deg


In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x_hist[1,1]/Re_mks-size_spacecraft[1]/2,x_hist[2,1]/Re_mks-size_spacecraft[2]/2,x_hist[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x_hist[1,1]/Re_mks-size_plate[1]/2,x_hist[2,1]/Re_mks-size_plate[2]/2,x_hist[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

sleep(5)
for i = 1:10:length(x_hist[1,:])-1
    for j = 0 : num_sat-1
        if j == 0
#             positions_1 = [x0_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            positions_1 = [x_hist[1,i]/Re_mks-origin_spacecraft[1],x_hist[2,i]/Re_mks-origin_spacecraft[2],x_hist[3,i]/Re_mks-origin_spacecraft[3]]
#             rotations_1 = i*2*pi/length(x0_soln[1+(6*j-6),:])
            position_1 = Translation(positions_1)
#             rotation_1 = LinearMap(RotZ(rotations_1))
            name_sat = "satellite"*string(j)
            settransform!(vis[name_sat], position_1)
#             settransform!(vis[name_sat], compose(position_1, rotation_1))
        end 
#         positions_2 = [x1_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
        positions_2 = [x_hist[7+(12*j),i]/Re_mks-origin_spacecraft[1],x_hist[8+(12*j),i]/Re_mks-origin_spacecraft[2],x_hist[9+(12*j),i]/Re_mks-origin_spacecraft[3]]
#         rotations_2 = i*2*pi/length(x1_soln[1+(6*j-6),:])
        position_2 = Translation(positions_2)
#         rotation_2 = LinearMap(RotZ(rotations_2))
        name_sat = "satellite"*string(j+1)
        settransform!(vis[name_sat], position_2)
#         settransform!(vis[name_sat], compose(position_2, rotation_2))
    end
        #     settransform!(vis["plate1"], position)
    sleep(0.05)
end


In [ ]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    r_noJ2 = x[7:9]
    v_noJ2 = x[10:12]

    pos_x = r[1]
    pos_y = r[2]
    pos_z = r[3]
    r_norm = sqrt(r[1]^2+r[2]^2+r[3]^2)
    r_norm_noJ2 = sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2)
    
    g = +1.5*J2*μ_mks*Re_mks^2/r_norm^4*[
        pos_x/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_y/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_z/r_norm*(5*pos_z^2/r_norm^2-3)] - μ_mks*r/r_norm^3
    
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = density_mks(r_norm - Re_mks)
    Fdrag = -0.5*(u*AoM_mks)*Cd*ρ*(sqrt(v_rel[1]^2+v_rel[2]^2+v_rel[3]^2))*v_rel

    v_rel_noJ2 = v_noJ2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ_noJ2 = density_mks(r_norm_noJ2 - Re_mks)
    Fdrag_noJ2 = -0.5*(u*AoM_mks)*Cd*ρ_noJ2*(sqrt(v_rel_noJ2[1]^2+v_rel_noJ2[2]^2+v_rel_noJ2[3]^2))*v_rel_noJ2

    v̇ = g + Fdrag
    v̇_noJ2 = -μ_mks/(r_norm_noJ2)^3 * r_noJ2 +  Fdrag_noJ2
    return [v; v̇; v_noJ2; v̇_noJ2]
end


In [ ]:
#Initial conditions in Cartesian Coordinates
# r0 = [a; 0; 0]
# v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
# x0 = [r0; v0; r0; v0];

# r1 = [a; 0; 0]
# v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
# x1 = [r1; v1; r1; v1];

# tspan = (0.0, Tfinal+50)
# Nt_t = Tfinal+150

r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

r1 = [a_mks; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ_mks/(a_mks)); sind(inc)*sqrt(μ_mks/(a_mks))]
x1 = [r1; v1; r1; v1];

step_t = 60*10 # save results every 10 mins
tspan = (0.0, Int(round((Tfinal)*P_mks/step_t)*step_t))
Nt_t = Int(round((Tfinal)*P_mks/step_t))-1#/100

In [ ]:
using Interpolations
#Full Orbit Mechanics Simulation and save results
# Nt = 150
# Nt = Nt*3
temp = LinRange(tspan[1],tspan[2],Nt_t)
x0_soln = zeros(6*num_sat,Nt_t)
x1_soln = zeros(6*num_sat,Nt_t)

x0_noJ2_soln = zeros(6*num_sat,Nt_t)
x1_noJ2_soln = zeros(6*num_sat,Nt_t)

Δa_soln = zeros(num_sat,Nt_t)
Δθ_soln = zeros(num_sat,Nt_t)
ΔΩ_soln = zeros(num_sat,Nt_t)

r_soln1 = zeros(num_sat,Nt_t)
ϕ_soln1 = zeros(num_sat,Nt_t)
θ_soln1 = zeros(num_sat,Nt_t)
u_soln1 = zeros(num_sat,Nt_t)

r_soln2 = zeros(num_sat,Nt_t)
ϕ_soln2 = zeros(num_sat,Nt_t)
θ_soln2 = zeros(num_sat,Nt_t)
u_soln2 = zeros(num_sat,Nt_t)

a_soln1 = zeros(num_sat,Nt_t)
ν_soln1 = zeros(num_sat,Nt_t)
Ω_soln1 = zeros(num_sat,Nt_t)

a_soln2 = zeros(num_sat,Nt_t)
ν_soln2 = zeros(num_sat,Nt_t)
Ω_soln2 = zeros(num_sat,Nt_t)

r_soln1_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln1_noJ2 = zeros(num_sat,Nt_t)
θ_soln1_noJ2 = zeros(num_sat,Nt_t)
u_soln1_noJ2 = zeros(num_sat,Nt_t)

r_soln2_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln2_noJ2 = zeros(num_sat,Nt_t)
θ_soln2_noJ2 = zeros(num_sat,Nt_t)
u_soln2_noJ2 = zeros(num_sat,Nt_t)

h_traj = 1#step_t#/P_mks
thist_c = thist.*P_mks
for i = 1:num_sat
    u1interp = 0;#LinearInterpolation(thist_c, u1hist[:,i])
    u2interp = 1;#LinearInterpolation(thist_c, u2hist[:,i])

    x_hist1 = zeros(length(x0),Int(tspan[2]/h_traj))
    x_hist2 = zeros(length(x1),Int(tspan[2]/h_traj))
    x_hist1[:,1] = x0 
    x_hist2[:,1] = x1 
    temp = LinRange(tspan[1],tspan[2],Nt_t)
    k = 0 
    save_index = 0
    for t = tspan[1]:h_traj:tspan[2]-step_t# temp# 1:100:tspan[2]-1
        k += 1
        #Stored Controller
        if t < tspan[2]
            u1 = 0;#u1interp(t) 
            u2 = 1;#u2interp(t) 
        else
            u1 = 0;#u_soln1[i,k]
            u2 = 0;#u_soln2[i,k]
        end

        x_hist1[:,k+1] .= sat_dynamics_rk4(x_hist1[:,k],u1,h_traj)
        x_hist2[:,k+1] .= sat_dynamics_rk4(x_hist2[:,k],u2,h_traj)

        # save results
        if k%step_t == 0
            save_index += 1
            u_soln1[i,save_index] = u1
            u_soln2[i,save_index] = u2
            x0_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[1:6,k]
            x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[7:12,k]
        
            x1_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[1:6,k]
            x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[7:12,k]
    
            rϕθ1_noJ2 = cartesian_to_spherical(x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2_noJ2 = cartesian_to_spherical(x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ1 = cartesian_to_spherical(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2 = cartesian_to_spherical(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            ΔaθΩ = cartesian_to_ΔθaΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]), mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ1 = cartesian_to_θaΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ2 = cartesian_to_θaΩ(mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
    
            Δθ_soln[i,save_index] = ΔaθΩ[1]    
            Δa_soln[i,save_index] = ΔaθΩ[2]
            ΔΩ_soln[i,save_index] = ΔaθΩ[3]

            ν_soln1[i,save_index] = aθΩ1[1]
            a_soln1[i,save_index] = aθΩ1[2]
            Ω_soln1[i,save_index] = aθΩ1[3]
        
            ν_soln2[i,save_index] = aθΩ2[1]
            a_soln2[i,save_index] = aθΩ2[2]
            Ω_soln2[i,save_index] = aθΩ2[3]
        
            r_soln1_noJ2[i,save_index] = rϕθ1_noJ2[1]
            ϕ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[2]
            θ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[3]
    
            r_soln2_noJ2[i,save_index] = rϕθ2_noJ2[1]
            ϕ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[2]
            θ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[3]
        
            r_soln1[i,save_index] = rϕθ1[1]
            ϕ_soln1[i,save_index] = rϕθ1[2]
            θ_soln1[i,save_index] = rϕθ1[3]
    
            r_soln2[i,save_index] = rϕθ2[1]
            ϕ_soln2[i,save_index] = rϕθ2[2]
            θ_soln2[i,save_index] = rϕθ2[3]
        end

    end
end
tsamp = LinRange(tspan[1],tspan[2],Nt_t);


In [ ]:
using Plots
# tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp[1:30:end]./(60*60*24),x0_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[3,1:30:end],color=:green)

plot!(tsamp[1:30:end]./(60*60*24),x1_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[3,1:30:end],color=:green)

# plot(tsamp,x0_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x0_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x0_noJ2_soln[3,:],color=:green)

# plot!(tsamp,x1_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x1_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x1_noJ2_soln[3,:],color=:green)
# x1_soln[3,:]

In [ ]:
plot(tsamp[1:30:end]./(60*60*24),broadcast.(rad2deg,ν_soln1[1,1:30:end]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln2[1,:]),color=:blue)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

# length(Ω_soln1[1,:][Ω_soln1[1,:] .>3.])

In [ ]:
tsamp = LinRange(tspan[1],tspan[2],Nt_t)
plot(tsamp[1:30:end]./(60*60*24),Δa_soln[1:30:end]',label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "Altitude Difference, km")


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Δθ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "True Anomaly Difference, deg")

In [ ]:
plot(tsamp./(60*40*24),broadcast.(rad2deg,ΔΩ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "RAAN Difference, deg", ylim = [-10,5])

In [ ]:
plot(tsamp,a_mks*ΔΩ_soln',label=permutedims(mylabel), ylim = [-100*1e1,10*1e2])
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
plot(tsamp./(60*60*24),u_soln1',color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),u_soln2',label=permutedims(mylabel2))
plot!(xlabel="Time, days", ylabel = "Control")


In [ ]:
plot(tsamp./(60*40*24),r_soln1_noJ2'.-Re_mks,color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*40*24),r_soln2_noJ2'.-Re_mks,label=permutedims(mylabel2))
for i=1:num_sat
    println(r_soln1_noJ2[1]-Re_mks," ",params.Δa_max*a_mks*i)
    plot!([tsamp[1]./(60*40*24), tsamp[end]./(60*40*24)],[r_soln1_noJ2[1]-Re_mks-params.Δa_max*a_mks*i, r_soln1_noJ2[1]-Re_mks[1]-params.Δa_max*a_mks*i],color=:red, linewidth=2,label="Limit")
end
plot!(xlabel= "Time, days",ylabel= "Altitude, km")


In [ ]:
plot(tsamp[1:30:end]./(60*40*24),ϕ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),ϕ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Azimuth, rad")

In [ ]:
plot(tsamp[1:30:end]./(60*40*24),θ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),θ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Elevation, rad")

In [ ]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [ ]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [ ]:
k1/k2

In [ ]:
(k1/k2)*2*pi

In [ ]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi